# RF-DETR Predictions Visualization on dataset2_split test

For each sequence in the test set, pick 4 frames and show:
1. Raw image
2. Image + GT bounding boxes
3. Image + predictions from `dataset2_augs`
4. Image + predictions from `correct_single_label`

In [ ]:
import re
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from collections import defaultdict

from rfdetr import RFDETRSmall

ROOT = Path('/home/dsa/stenosis')
TEST_DIR = ROOT / 'data' / 'dataset2_split' / 'test'
IMG_DIR = TEST_DIR / 'images'
LBL_DIR = TEST_DIR / 'labels'

CKPT_DATASET2 = ROOT / 'rfdetr_runs' / 'dataset2_augs' / 'checkpoint_best_total.pth'
CKPT_CORRECT = ROOT / 'rfdetr_runs' / 'correct_single_labelf_rfdetr_arcade_dataset2' / 'checkpoint_best_total.pth'

FRAMES_PER_SEQ = 4
CONF_THRESHOLD = 0.15

In [ ]:
# Group images by sequence
seq_pattern = re.compile(r'^(\d+_\d+_\d+)_\d+_bmp_jpg')
sequences = defaultdict(list)

for img_path in sorted(IMG_DIR.glob('*.jpg')):
    m = seq_pattern.match(img_path.name)
    if m:
        sequences[m.group(1)].append(img_path)

print(f'Found {len(sequences)} sequences')
for seq, imgs in sorted(sequences.items()):
    print(f'  {seq}: {len(imgs)} frames')

In [ ]:
# Select frames: pick evenly spaced frames from each sequence
selected_frames = []
for seq_id in sorted(sequences.keys()):
    imgs = sequences[seq_id]
    n = len(imgs)
    if n <= FRAMES_PER_SEQ:
        selected_frames.extend(imgs)
    else:
        indices = np.linspace(0, n - 1, FRAMES_PER_SEQ, dtype=int)
        selected_frames.extend([imgs[i] for i in indices])

print(f'Selected {len(selected_frames)} frames total')

In [ ]:
def load_rfdetr(ckpt_path):
    model = RFDETRSmall()
    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    nc = ckpt['model']['class_embed.bias'].shape[0]
    model.model.model.reinitialize_detection_head(nc)
    model.model.model.load_state_dict(ckpt['model'], strict=True)
    model.model.model.to('cuda')
    model.model.model.eval()
    return model

print('Loading dataset2_augs model...')
model_ds2 = load_rfdetr(CKPT_DATASET2)
print('Loading correct_single_label model...')
model_csl = load_rfdetr(CKPT_CORRECT)
print('Done!')

In [ ]:
def parse_yolo_label(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists() or label_path.stat().st_size == 0:
        return boxes
    for line in label_path.read_text().strip().split('\n'):
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        x1 = (cx - w / 2) * img_w
        y1 = (cy - h / 2) * img_h
        x2 = (cx + w / 2) * img_w
        y2 = (cy + h / 2) * img_h
        boxes.append((x1, y1, x2, y2))
    return boxes


def draw_boxes(ax, boxes, color='lime', linewidth=2, label=None):
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=linewidth, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        if label and i == 0:
            ax.text(x1, y1 - 4, label, color=color, fontsize=7,
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.6))


def draw_detections(ax, detections, color='red', threshold=CONF_THRESHOLD):
    if detections is None or len(detections) == 0:
        return
    mask = detections.confidence >= threshold
    for xyxy, conf in zip(detections.xyxy[mask], detections.confidence[mask]):
        x1, y1, x2, y2 = xyxy
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f'{conf:.2f}', color=color, fontsize=7,
                bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.6))

In [ ]:
# Run predictions on all selected frames
print(f'Running inference on {len(selected_frames)} frames...')

preds_ds2 = {}
preds_csl = {}

for i, img_path in enumerate(selected_frames):
    preds_ds2[img_path.name] = model_ds2.predict(str(img_path), threshold=CONF_THRESHOLD)
    preds_csl[img_path.name] = model_csl.predict(str(img_path), threshold=CONF_THRESHOLD)
    if (i + 1) % 20 == 0:
        print(f'  {i + 1}/{len(selected_frames)}')

print('Inference done!')

In [ ]:
# Visualize
for img_path in selected_frames:
    img = Image.open(img_path)
    img_np = np.array(img)
    img_w, img_h = img.size

    lbl_path = LBL_DIR / (img_path.stem + '.txt')
    gt_boxes = parse_yolo_label(lbl_path, img_w, img_h)

    det_ds2 = preds_ds2[img_path.name]
    det_csl = preds_csl[img_path.name]

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    # 1. Raw image
    axes[0].imshow(img_np)
    axes[0].set_title('Raw', fontsize=11)
    axes[0].axis('off')

    # 2. GT
    axes[1].imshow(img_np)
    draw_boxes(axes[1], gt_boxes, color='lime', label='GT')
    axes[1].set_title(f'GT ({len(gt_boxes)} boxes)', fontsize=11)
    axes[1].axis('off')

    # 3. dataset2_augs preds
    n_ds2 = int((det_ds2.confidence >= CONF_THRESHOLD).sum()) if det_ds2 is not None and len(det_ds2) > 0 else 0
    axes[2].imshow(img_np)
    draw_boxes(axes[2], gt_boxes, color='lime', linewidth=1)
    draw_detections(axes[2], det_ds2, color='red')
    axes[2].set_title(f'dataset2_augs ({n_ds2} preds)', fontsize=11)
    axes[2].axis('off')

    # 4. correct_single_label preds
    n_csl = int((det_csl.confidence >= CONF_THRESHOLD).sum()) if det_csl is not None and len(det_csl) > 0 else 0
    axes[3].imshow(img_np)
    draw_boxes(axes[3], gt_boxes, color='lime', linewidth=1)
    draw_detections(axes[3], det_csl, color='cyan')
    axes[3].set_title(f'correct_single_label ({n_csl} preds)', fontsize=11)
    axes[3].axis('off')

    m = seq_pattern.match(img_path.name)
    seq_id = m.group(1) if m else '?'
    fig.suptitle(f'Sequence {seq_id} \u2014 {img_path.name}', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()